# 01 - Esplorazione Preliminare del Dataset
Analisi esplorativa del dataset T1DiabetesGranada (Sezione 3.2 della tesi).

In [ ]:
import sys, os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/t1dbg'
except ImportError:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

## Caricamento dati

In [ ]:
from lib.preprocessing import load_data
from lib.config import get_raw_file

df = load_data(get_raw_file("Glucose_measurements.csv"))

In [ ]:
print(f"Shape: {df.shape}")
print(f"Pazienti unici: {df['Patient_ID'].nunique()}")
df.describe()

## Distribuzione dei valori glicemici
Distribuzione complessiva delle misurazioni CGM per verificare la concentrazione nel range normoglicemico (70-180 mg/dL).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df['Measurement'], bins=50, edgecolor='black', alpha=0.7)
ax.set_xlabel('Glicemia (mg/dL)')
ax.set_ylabel('Frequenza (%)')
ax.set_title('Distribuzione delle misurazioni glicemiche')
plt.tight_layout()
plt.show()

## Box plot per paziente
Variabilita inter-paziente delle misurazioni glicemiche su un campione casuale di 10 pazienti.

In [ ]:
import random
random.seed(42)
sample_patients = random.sample(list(df['Patient_ID'].unique()), 10)
sample_df = df[df['Patient_ID'].isin(sample_patients)]

fig, ax = plt.subplots(figsize=(12, 6))
sample_df.boxplot(column='Measurement', by='Patient_ID', ax=ax)
ax.set_xlabel('Pazienti')
ax.set_ylabel('Glicemia (mg/dL)')
ax.set_title('Box plot per 10 pazienti scelti casualmente')
plt.suptitle('')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Funzione di autocorrelazione (ACF)
L'ACF misura la correlazione della serie glicemica con i propri ritardi, utile per verificare la dipendenza temporale che motiva l'uso di finestre scorrevoli.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

sample_patients_acf = random.sample(list(df['Patient_ID'].unique()), 2)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, pid in enumerate(sample_patients_acf):
    patient_data = df[df['Patient_ID'] == pid]['Measurement'].dropna()
    plot_acf(patient_data, lags=50, ax=axes[i], title=f'ACF - Paziente {pid}')
plt.tight_layout()
plt.show()

## Funzione di autocorrelazione parziale (PACF)
La PACF isola il contributo diretto di ciascun ritardo, utile per determinare il numero di lag da includere nelle finestre (HL=8, corrispondente a 2 ore).

In [ ]:
sample_patients_pacf = random.sample(list(df['Patient_ID'].unique()), 4)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for i, pid in enumerate(sample_patients_pacf):
    ax = axes[i // 2][i % 2]
    patient_data = df[df['Patient_ID'] == pid]['Measurement'].dropna()
    plot_pacf(patient_data, lags=50, ax=ax, title=f'PACF - Paziente {pid}')
plt.tight_layout()
plt.show()